# Demo 2 — O Limite do Contexto Curto

## O que vamos fazer aqui?

No Demo 1 construímos um modelo de bigramas que **prevê a próxima palavra olhando apenas para a palavra anterior**.

Neste notebook vamos mostrar **onde esse modelo falha** — e *por que* a falha é estrutural, não um detalhe que se corrige com mais dados.

---

## O problema central

Considere estas duas frases:

> *"A fazenda teve **estiagem severa**, então a produtividade do milho **caiu**."*  
> *"A fazenda teve **chuva regular**, então a produtividade do milho **subiu**."*

A última palavra (`caiu` / `subiu`) depende de uma palavra que apareceu **8 posições antes** (`estiagem` / `chuva`).

Um modelo que só olha para a última palavra vê apenas `"milho"` antes de decidir — e não tem como saber se veio de um contexto de estiagem ou de chuva.

Isso não é um bug. É uma **limitação arquitetural**: o modelo não tem onde guardar essa informação.

---

## O que vamos explorar

1. Reproduzir o problema com o modelo de bigramas do Demo 1
2. Construir modelos com janelas de contexto maiores (2, 3, 5 palavras)
3. Mostrar que janelas maiores **ajudam** mas **não resolvem**
4. Visualizar a "memória que desaparece" à medida que olhamos só para trás
5. Concluir: o problema não era prever a próxima palavra — era **carregar contexto útil**

---

> **Frase para guardar:**  
> *"O problema não era só prever palavra — era carregar contexto útil."*

---
## Parte 1 — Configuração inicial

Reutilizamos exatamente o mesmo código do Demo 1. Em produção, isso seria importado de um módulo. Aqui repetimos para o notebook ser autocontido.

In [1]:
import re
import random
from collections import defaultdict, Counter

# ── tokenização ────────────────────────────────────────────────────────────
def tokenizar(frase):
    """Converte frase em lista de tokens: minúsculas, sem pontuação."""
    frase = frase.lower()
    frase = re.sub(r'[^a-záàâãéêíóôõúüç\s]', '', frase)
    return [t for t in frase.split() if t]


# ── modelo de n-gramas com janela variável ─────────────────────────────────
def construir_modelo_ngrama(corpus, n=1):
    """
    Constrói um modelo de n-gramas com janela de contexto de tamanho n.

    n=1 → bigrama   (contexto: 1 palavra)
    n=2 → trigrama  (contexto: 2 palavras)
    n=3 → 4-grama   (contexto: 3 palavras)
    etc.

    A chave do dicionário é uma TUPLA com as últimas n palavras vistas.
    O valor é uma lista com as próximas palavras observadas.

    Args:
        corpus (list[str]): Lista de frases.
        n (int): Tamanho da janela de contexto.

    Returns:
        defaultdict(list): Modelo n-grama.
    """
    modelo = defaultdict(list)

    for frase in corpus:
        tokens = tokenizar(frase)

        # Precisamos de ao menos n+1 tokens para extrair um n-grama
        if len(tokens) < n + 1:
            continue

        # Deslizamos uma janela de n tokens ao longo da frase
        # Para cada posição i, o contexto é tokens[i:i+n]
        # e a próxima palavra é tokens[i+n]
        for i in range(len(tokens) - n):
            contexto = tuple(tokens[i : i + n])   # janela de n palavras
            proxima  = tokens[i + n]               # palavra que vem depois
            modelo[contexto].append(proxima)

    return modelo


# ── geração com janela variável ────────────────────────────────────────────
def gerar_frase_ngrama(modelo, semente, n, tamanho_maximo=20):
    """
    Gera uma frase usando um modelo de n-gramas.

    Args:
        modelo: O modelo n-grama (dict).
        semente (list[str]): Lista com as n primeiras palavras da frase.
                             Deve ter exatamente n palavras.
        n (int): Tamanho da janela de contexto.
        tamanho_maximo (int): Limite de palavras na frase gerada.

    Returns:
        str: Frase gerada.
    """
    if len(semente) != n:
        raise ValueError(f"A semente deve ter exatamente {n} palavra(s).")

    frase = list(semente)             # começa com as palavras-semente
    contexto = tuple(semente)         # janela de contexto inicial

    for _ in range(tamanho_maximo - n):
        if contexto not in modelo:
            break   # beco sem saída — nunca vimos esse contexto no corpus

        proxima = random.choice(modelo[contexto])
        frase.append(proxima)

        # Desliza a janela: descarta a palavra mais antiga, inclui a nova
        contexto = tuple(frase[-n:])

    return " ".join(frase)


print("Funções carregadas. Prontos para o Demo 2.")

Funções carregadas. Prontos para o Demo 2.


---
## Parte 2 — Corpus com Dependências Longas

Para este demo, precisamos de um corpus onde **a palavra correta no final da frase depende de uma palavra no começo**.

Vamos usar pares de frases contrastantes:
- Frase A: contexto de **estiagem** → produtividade **caiu**
- Frase B: contexto de **chuva**    → produtividade **subiu**

A estrutura intermediária é idêntica — só o contexto inicial muda.

Isso torna o problema de dependência longa **cirúrgico e verificável**.

In [2]:
# Corpus especialmente projetado para expor dependências longas.
#
# Padrão intencional:
#   [CAUSA no começo] ... estrutura intermediária comum ... [EFEITO no final]
#
# Para um modelo com janela curta, o efeito no final parece imprevisível
# porque a causa já "saiu" da janela de contexto.

corpus_dependencia = [
    # ── par 1: estiagem → caiu / chuva → subiu ───────────────────────────
    # A causa está na posição 4 ("estiagem"), o efeito na posição 11 ("caiu")
    # Distância: 7 palavras
    "a fazenda teve estiagem severa entao a produtividade do milho caiu",
    "a fazenda teve estiagem severa entao a produtividade do milho caiu",
    "a fazenda teve estiagem severa entao a produtividade do milho caiu",
    "a fazenda teve chuva regular entao a produtividade do milho subiu",
    "a fazenda teve chuva regular entao a produtividade do milho subiu",
    "a fazenda teve chuva regular entao a produtividade do milho subiu",

    # ── par 2: seca → prejudicou / irrigação → beneficiou ────────────────
    "o campo passou por seca prolongada e a colheita da soja prejudicou muito",
    "o campo passou por seca prolongada e a colheita da soja prejudicou muito",
    "o campo passou por irrigacao adequada e a colheita da soja beneficiou muito",
    "o campo passou por irrigacao adequada e a colheita da soja beneficiou muito",

    # ── par 3: geada → perdemos / clima bom → ganhamos ───────────────────
    "neste ano o clima teve geada forte e no final da safra perdemos bastante",
    "neste ano o clima teve geada forte e no final da safra perdemos bastante",
    "neste ano o clima teve temperatura boa e no final da safra ganhamos bastante",
    "neste ano o clima teve temperatura boa e no final da safra ganhamos bastante",

    # ── par 4: praga → diminuiu / sem praga → aumentou ───────────────────
    "a lavoura de cafe sofreu com praga severa portanto a producao do ano diminuiu",
    "a lavoura de cafe sofreu com praga severa portanto a producao do ano diminuiu",
    "a lavoura de cafe ficou livre de praga portanto a producao do ano aumentou",
    "a lavoura de cafe ficou livre de praga portanto a producao do ano aumentou",

    # ── frases extras para enriquecer o vocabulário ───────────────────────
    "a produtividade do milho depende do solo e do clima",
    "a colheita da soja foi boa neste ano",
    "a fazenda investiu em irrigacao para proteger a lavoura",
    "o produtor monitorou o campo durante toda a safra",
    "a estiagem afetou varias regioes do pais neste ano",
    "a chuva chegou no momento certo e salvou a colheita",
]

print(f"Corpus criado com {len(corpus_dependencia)} frases.")
print()
print("Frases-chave (pares contrastantes):")
print("  ESTIAGEM → " + corpus_dependencia[0])
print("  CHUVA    → " + corpus_dependencia[3])

Corpus criado com 24 frases.

Frases-chave (pares contrastantes):
  ESTIAGEM → a fazenda teve estiagem severa entao a produtividade do milho caiu
  CHUVA    → a fazenda teve chuva regular entao a produtividade do milho subiu


---
## Parte 3 — O Experimento Central

Vamos treinar modelos com janelas de diferentes tamanhos e fazer a mesma pergunta para todos:

> **Dado o trecho `"...a produtividade do milho"`, qual palavra vem a seguir?**

A resposta correta **depende do contexto do começo da frase**:
- Se a frase começou com `"estiagem"` → deve vir `"caiu"`
- Se a frase começou com `"chuva"` → deve vir `"subiu"`

Vamos ver até que ponto cada janela consegue capturar essa informação.

In [3]:
# Treina modelos com janelas de 1 a 6 palavras
janelas = [1, 2, 3, 4, 5, 6]
modelos = {}

for n in janelas:
    modelos[n] = construir_modelo_ngrama(corpus_dependencia, n=n)
    n_contextos = len(modelos[n])
    print(f"Janela {n}: {n_contextos:>4} contextos únicos aprendidos")

print()
print("Quanto maior a janela, mais contextos únicos — e mais esparsidade.")
print("Com corpus pequeno, janelas grandes rapidamente ficam sem dados.")

Janela 1:   66 contextos únicos aprendidos
Janela 2:   95 contextos únicos aprendidos
Janela 3:   93 contextos únicos aprendidos
Janela 4:   89 contextos únicos aprendidos
Janela 5:   84 contextos únicos aprendidos
Janela 6:   75 contextos únicos aprendidos

Quanto maior a janela, mais contextos únicos — e mais esparsidade.
Com corpus pequeno, janelas grandes rapidamente ficam sem dados.


In [4]:
def consultar_modelo(modelo, contexto_tuple):
    """
    Consulta o modelo para um contexto específico e retorna a distribuição
    de probabilidades das próximas palavras.

    Args:
        modelo: O modelo n-grama.
        contexto_tuple (tuple): Tupla de palavras de contexto.

    Returns:
        dict: {palavra: probabilidade} ou {} se contexto desconhecido.
    """
    if contexto_tuple not in modelo:
        return {}   # nunca vimos esse contexto

    candidatos = modelo[contexto_tuple]
    total = len(candidatos)
    contagem = Counter(candidatos)

    return {palavra: n / total for palavra, n in contagem.most_common()}


# ── Experimento: cada modelo, mesma pergunta ────────────────────────────────
#
# Pergunta: "depois de 'milho', qual palavra vem?"
# O contexto varia conforme o tamanho da janela de cada modelo.
#
# Para tornar a comparação justa, reconstruímos o contexto como seriam
# as últimas n palavras antes de "caiu"/"subiu" na frase original:
#
#   frase: "a fazenda teve estiagem severa entao a produtividade do milho ___"
#   posição da lacuna: 11
#   janela 1: ("milho",)
#   janela 2: ("do", "milho")
#   janela 3: ("produtividade", "do", "milho")
#   janela 4: ("a", "produtividade", "do", "milho")
#   janela 5: ("entao", "a", "produtividade", "do", "milho")
#   janela 6: ("severa", "entao", "a", "produtividade", "do", "milho")
#   janela 7: ("estiagem", "severa", "entao", "a", "produtividade", "do", "milho")
#             ^ aqui aparece a causa!

frase_estiagem = tokenizar("a fazenda teve estiagem severa entao a produtividade do milho caiu")
frase_chuva    = tokenizar("a fazenda teve chuva regular entao a produtividade do milho subiu")

# Posição onde a palavra-alvo aparece (índice 10 = "caiu"/"subiu")
posicao_alvo = 10

print("=" * 65)
print("EXPERIMENTO: O que o modelo prevê antes de 'caiu'/'subiu'?")
print("=" * 65)

for n in janelas:
    # Extraímos as últimas n palavras antes da posição alvo
    contexto_estiagem = tuple(frase_estiagem[posicao_alvo - n : posicao_alvo])
    contexto_chuva    = tuple(frase_chuva   [posicao_alvo - n : posicao_alvo])

    probs_estiagem = consultar_modelo(modelos[n], contexto_estiagem)
    probs_chuva    = consultar_modelo(modelos[n], contexto_chuva)

    print(f"\n─── Janela = {n} palavra(s) ───")
    print(f"  Contexto (estiagem): {list(contexto_estiagem)}")
    print(f"  Contexto (chuva)  : {list(contexto_chuva)}")

    # Mostramos a probabilidade de 'caiu' e 'subiu' em cada caso
    p_caiu_estiagem  = probs_estiagem.get("caiu",  0)
    p_subiu_estiagem = probs_estiagem.get("subiu", 0)
    p_caiu_chuva     = probs_chuva.get("caiu",    0)
    p_subiu_chuva    = probs_chuva.get("subiu",   0)

    def barra(p): return "█" * int(p * 20)

    print(f"  P(caiu  | contexto estiagem) = {p_caiu_estiagem:.0%}  {barra(p_caiu_estiagem)}")
    print(f"  P(subiu | contexto estiagem) = {p_subiu_estiagem:.0%}  {barra(p_subiu_estiagem)}")
    print(f"  P(caiu  | contexto chuva)    = {p_caiu_chuva:.0%}  {barra(p_caiu_chuva)}")
    print(f"  P(subiu | contexto chuva)    = {p_subiu_chuva:.0%}  {barra(p_subiu_chuva)}")

    # Diagnóstico: o modelo consegue distinguir os dois contextos?
    distingue = (p_caiu_estiagem != p_caiu_chuva) or (probs_estiagem != probs_chuva and probs_estiagem and probs_chuva)
    if not probs_estiagem or not probs_chuva:
        status = "⚠  contexto não visto no corpus (sem dados)"
    elif distingue:
        status = "✓  modelo DISTINGUE os dois contextos"
    else:
        status = "✗  modelo NÃO distingue — responde igual para os dois"
    print(f"  → {status}")

EXPERIMENTO: O que o modelo prevê antes de 'caiu'/'subiu'?

─── Janela = 1 palavra(s) ───
  Contexto (estiagem): ['milho']
  Contexto (chuva)  : ['milho']
  P(caiu  | contexto estiagem) = 43%  ████████
  P(subiu | contexto estiagem) = 43%  ████████
  P(caiu  | contexto chuva)    = 43%  ████████
  P(subiu | contexto chuva)    = 43%  ████████
  → ✗  modelo NÃO distingue — responde igual para os dois

─── Janela = 2 palavra(s) ───
  Contexto (estiagem): ['do', 'milho']
  Contexto (chuva)  : ['do', 'milho']
  P(caiu  | contexto estiagem) = 43%  ████████
  P(subiu | contexto estiagem) = 43%  ████████
  P(caiu  | contexto chuva)    = 43%  ████████
  P(subiu | contexto chuva)    = 43%  ████████
  → ✗  modelo NÃO distingue — responde igual para os dois

─── Janela = 3 palavra(s) ───
  Contexto (estiagem): ['produtividade', 'do', 'milho']
  Contexto (chuva)  : ['produtividade', 'do', 'milho']
  P(caiu  | contexto estiagem) = 43%  ████████
  P(subiu | contexto estiagem) = 43%  ████████
  P(caiu 

---
## Parte 4 — Visualizando a Memória que Desaparece

Vamos tornar visível o momento exato em que cada janela "alcança" a palavra causadora.

Pense em uma lanterna que ilumina apenas `n` palavras para trás. À medida que avançamos na frase, as palavras do começo entram e saem do foco da lanterna.

O modelo só pode usar o que está **dentro da lanterna**.

In [5]:
def visualizar_janela_deslizante(frase_tokens, posicao_alvo, janelas):
    """
    Mostra, para cada tamanho de janela, quais palavras estão visíveis
    no momento em que o modelo precisa prever a palavra na posição_alvo.

    Usa caracteres visuais para indicar:
      [palavra]  → dentro da janela (visível para o modelo)
       palavra   → fora da janela (invisível — já "esquecido")

    Args:
        frase_tokens (list[str]): Tokens da frase.
        posicao_alvo (int): Índice da palavra a prever.
        janelas (list[int]): Lista de tamanhos de janela a comparar.
    """
    frase_str = " ".join(frase_tokens[:posicao_alvo]) + " [___]"
    print(f"Frase: {frase_str}")
    print(f"       (o modelo precisa prever [___] = posição {posicao_alvo})")
    print()

    for n in janelas:
        # Índice de início da janela
        inicio_janela = posicao_alvo - n

        partes = []
        for i, token in enumerate(frase_tokens[:posicao_alvo]):
            if i >= inicio_janela:
                # Dentro da janela: destacado com colchetes
                partes.append(f"[{token}]")
            else:
                # Fora da janela: apagado com pontos
                partes.append("." * len(token))

        linha = " ".join(partes) + " ?"
        print(f"  janela={n}: {linha}")

        # Aponta se a palavra-causa está visível
        causa_visivel = inicio_janela <= 3  # "estiagem" está na posição 3
        if causa_visivel:
            print(f"           ^ CAUSA visível na janela!")

    print()


print("=" * 65)
print("FRASE COM ESTIAGEM — o que cada janela enxerga")
print("=" * 65)
visualizar_janela_deslizante(frase_estiagem, posicao_alvo=10, janelas=[1, 2, 3, 4, 5, 6, 7, 8])

print("=" * 65)
print("FRASE COM CHUVA — o que cada janela enxerga")
print("=" * 65)
visualizar_janela_deslizante(frase_chuva, posicao_alvo=10, janelas=[1, 2, 3, 4, 5, 6, 7, 8])

FRASE COM ESTIAGEM — o que cada janela enxerga
Frase: a fazenda teve estiagem severa entao a produtividade do milho [___]
       (o modelo precisa prever [___] = posição 10)

  janela=1: . ....... .... ........ ...... ..... . ............. .. [milho] ?
  janela=2: . ....... .... ........ ...... ..... . ............. [do] [milho] ?
  janela=3: . ....... .... ........ ...... ..... . [produtividade] [do] [milho] ?
  janela=4: . ....... .... ........ ...... ..... [a] [produtividade] [do] [milho] ?
  janela=5: . ....... .... ........ ...... [entao] [a] [produtividade] [do] [milho] ?
  janela=6: . ....... .... ........ [severa] [entao] [a] [produtividade] [do] [milho] ?
  janela=7: . ....... .... [estiagem] [severa] [entao] [a] [produtividade] [do] [milho] ?
           ^ CAUSA visível na janela!
  janela=8: . ....... [teve] [estiagem] [severa] [entao] [a] [produtividade] [do] [milho] ?
           ^ CAUSA visível na janela!

FRASE COM CHUVA — o que cada janela enxerga
Frase: a fazenda teve ch

---
## Parte 5 — O Problema da Ambiguidade Referencial

Há um segundo tipo de falha, talvez ainda mais interessante: a **ambiguidade pronominal**.

Considere:

> *"O produtor visitou a fazenda depois da chuva porque **ela** havia alagado a área."*

A palavra `"ela"` se refere a `"chuva"` — mas a referência está **5 posições antes**.

Para resolver esse tipo de ambiguidade, o modelo precisa manter uma representação de **qual entidade está sendo referenciada** ao longo de toda a frase. Isso é exatamente o que o mecanismo de atenção do transformer faz.

In [6]:
# Vamos montar um mini corpus com frases que contêm pronomes ambíguos
# e ver o que cada modelo de janela prevê depois do pronome

corpus_pronomes = [
    # Referência de "ela" → chuva (fenômeno climático, causa de alagamento)
    "o produtor visitou a fazenda depois da chuva porque ela havia alagado a area",
    "o campo foi inspecionado depois da chuva porque ela tinha danificado as plantas",
    "a lavoura foi vistoriada apos a chuva porque ela destruiu parte da colheita",

    # Referência de "ela" → estiagem (fenômeno climático, causa de ressecamento)
    "a fazenda sofreu com a estiagem porque ela secou todos os rios da regiao",
    "o produtor preocupou se com a estiagem porque ela afetou o solo profundamente",

    # Referência de "ele" → produtor (sujeito humano, agente de ação)
    "a fazenda pertence ao produtor e ele cuida de tudo diariamente",
    "o campo foi preparado pelo produtor porque ele conhece bem o solo",
    "o engenheiro agronomo visitou o produtor porque ele pediu uma consultoria",

    # Frases extras para enriquecer
    "ela havia causado muito dano a plantacao",
    "ela secou o solo completamente",
    "ele decidiu investir em irrigacao",
    "ele plantou soja nesta safra",
]

# Treina modelo de bigramas (janela 1)
modelo_bigrama_pronomes = construir_modelo_ngrama(corpus_pronomes, n=1)

# Treina modelo de 4-gramas (janela 3)
modelo_4grama_pronomes = construir_modelo_ngrama(corpus_pronomes, n=3)

print("=" * 65)
print("PROBLEMA DO PRONOME: o que o modelo prevê depois de 'ela'?")
print("=" * 65)
print()

# Com janela 1: o modelo só sabe que veio "ela" — sem contexto nenhum
probs_ela_janela1 = consultar_modelo(modelo_bigrama_pronomes, ("ela",))

print("Janela 1 (só a palavra 'ela'):")
if probs_ela_janela1:
    for palavra, prob in sorted(probs_ela_janela1.items(), key=lambda x: -x[1]):
        barra = "█" * int(prob * 30)
        print(f"  P('{palavra:<12}' | ela) = {prob:.0%}  {barra}")
else:
    print("  Sem dados.")

print()
print("→ Com janela 1, 'ela' pode continuar de qualquer jeito.")
print("  O modelo não sabe se 'ela' se refere a chuva, estiagem, ou outra coisa.")

print()
print("─" * 65)
print()

# Com janela 3: o modelo vê as 3 palavras antes de qualquer coisa que vem depois de "ela"
# Aqui, se o contexto for "da chuva porque ela", o modelo pode aprender padrões mais específicos
contexto_chuva_ela    = ("porque", "ela")
contexto_estiagem_ela = ("porque", "ela")

# Nota: com janela 2, ambos os contextos terminam em ("porque", "ela")
# só janela maior capturaria a diferença entre chuva e estiagem

contexto_chuva_4g    = ("da", "chuva", "porque")
contexto_estiagem_4g = ("da", "estiagem", "porque")

probs_chuva_4g    = consultar_modelo(modelo_4grama_pronomes, contexto_chuva_4g)
probs_estiagem_4g = consultar_modelo(modelo_4grama_pronomes, contexto_estiagem_4g)

print("Janela 3 — contexto 'da chuva porque' (prevendo depois de 'porque'):")
if probs_chuva_4g:
    for palavra, prob in sorted(probs_chuva_4g.items(), key=lambda x: -x[1]):
        print(f"  P('{palavra:<12}' | da chuva porque) = {prob:.0%}")
else:
    print("  Contexto não visto no corpus.")

print()
print("Janela 3 — contexto 'da estiagem porque' (prevendo depois de 'porque'):")
if probs_estiagem_4g:
    for palavra, prob in sorted(probs_estiagem_4g.items(), key=lambda x: -x[1]):
        print(f"  P('{palavra:<12}' | da estiagem porque) = {prob:.0%}")
else:
    print("  Contexto não visto no corpus.")

PROBLEMA DO PRONOME: o que o modelo prevê depois de 'ela'?

Janela 1 (só a palavra 'ela'):
  P('havia       ' | ela) = 29%  ████████
  P('secou       ' | ela) = 29%  ████████
  P('tinha       ' | ela) = 14%  ████
  P('destruiu    ' | ela) = 14%  ████
  P('afetou      ' | ela) = 14%  ████

→ Com janela 1, 'ela' pode continuar de qualquer jeito.
  O modelo não sabe se 'ela' se refere a chuva, estiagem, ou outra coisa.

─────────────────────────────────────────────────────────────────

Janela 3 — contexto 'da chuva porque' (prevendo depois de 'porque'):
  P('ela         ' | da chuva porque) = 100%

Janela 3 — contexto 'da estiagem porque' (prevendo depois de 'porque'):
  Contexto não visto no corpus.


---
## Parte 6 — A Esparsidade: o Preço de Janelas Maiores

Há um problema prático com janelas maiores: **esparsidade**.

Quanto maior a janela, mais combinações únicas de contexto existem — e mais difícil é ter visto todas elas no corpus.

Com corpus pequeno, janelas grandes simplesmente **não têm dados suficientes** para funcionar.

In [7]:
def calcular_cobertura(modelo, corpus, n):
    """
    Calcula qual porcentagem dos contextos encontrados no corpus
    estão representados no modelo.

    Um contexto "não coberto" significa que o modelo não tem dados
    para fazer previsão naquele ponto — e precisaria de um fallback.

    Args:
        modelo: O modelo n-grama.
        corpus (list[str]): O corpus de frases.
        n (int): Tamanho da janela.

    Returns:
        tuple: (total de contextos, contextos cobertos, porcentagem)
    """
    total   = 0
    coberto = 0

    for frase in corpus:
        tokens = tokenizar(frase)
        for i in range(n, len(tokens)):
            contexto = tuple(tokens[i - n : i])
            total += 1
            if contexto in modelo:
                coberto += 1

    pct = coberto / total * 100 if total > 0 else 0
    return total, coberto, pct


print("Cobertura do modelo vs tamanho da janela")
print("(quanto do corpus cada modelo consegue 'prever')")
print()
print(f"{'Janela':<10} {'Contextos':<15} {'Cobertos':<15} {'Cobertura':>10}  Barra")
print("-" * 65)

for n in janelas:
    total, coberto, pct = calcular_cobertura(modelos[n], corpus_dependencia, n)
    barra = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
    print(f"n={n:<8} {total:<15} {coberto:<15} {pct:>9.1f}%  {barra}")

print()
print("Observe: janelas maiores têm menos cobertura.")
print("Com um corpus real de terabytes, esse problema desaparece —")
print("mas a complexidade de armazenamento explode exponencialmente.")

Cobertura do modelo vs tamanho da janela
(quanto do corpus cada modelo consegue 'prever')

Janela     Contextos       Cobertos         Cobertura  Barra
-----------------------------------------------------------------
n=1        261             261                 100.0%  ████████████████████
n=2        237             237                 100.0%  ████████████████████
n=3        213             213                 100.0%  ████████████████████
n=4        189             189                 100.0%  ████████████████████
n=5        165             165                 100.0%  ████████████████████
n=6        141             141                 100.0%  ████████████████████

Observe: janelas maiores têm menos cobertura.
Com um corpus real de terabytes, esse problema desaparece —
mas a complexidade de armazenamento explode exponencialmente.


---
## Parte 7 — Comparação de Frases Geradas por Janelas Diferentes

Vamos fazer algo intuitivo: gerar frases com diferentes tamanhos de janela e observar a qualidade.

- Janelas pequenas → frases gramaticalmente plausíveis localmente, mas incoerentes globalmente
- Janelas grandes → frases mais coerentes, mas frequentemente "travam" por falta de contexto visto

In [8]:
random.seed(42)

# Corpus unificado para geração
corpus_unificado = corpus_dependencia[:]

print("Frases geradas com diferentes tamanhos de janela")
print("Semente: início da frase do corpus")
print()

for n in [1, 2, 3, 4]:
    modelo_n = construir_modelo_ngrama(corpus_unificado, n=n)

    # Usamos as n primeiras palavras do corpus como semente
    semente_base = tokenizar("a fazenda teve estiagem")
    semente = semente_base[:n]  # pegamos apenas as primeiras n palavras

    print(f"─── Janela n={n} ─── semente: {semente}")
    for i in range(4):
        try:
            frase = gerar_frase_ngrama(modelo_n, semente, n=n, tamanho_maximo=14)
            print(f"  {i+1}. {frase}")
        except Exception as e:
            print(f"  {i+1}. [erro: {e}]")
    print()

Frases geradas com diferentes tamanhos de janela
Semente: início da frase do corpus

─── Janela n=1 ─── semente: ['a']
  1. a produtividade do solo e a colheita da soja foi boa neste ano o
  2. a produtividade do milho caiu
  3. a produtividade do milho subiu
  4. a fazenda teve geada forte e a lavoura de cafe sofreu com praga severa

─── Janela n=2 ─── semente: ['a', 'fazenda']
  1. a fazenda teve chuva regular entao a produtividade do milho caiu
  2. a fazenda teve chuva regular entao a produtividade do milho depende do solo e
  3. a fazenda teve chuva regular entao a produtividade do milho subiu
  4. a fazenda teve chuva regular entao a produtividade do milho subiu

─── Janela n=3 ─── semente: ['a', 'fazenda', 'teve']
  1. a fazenda teve estiagem severa entao a produtividade do milho caiu
  2. a fazenda teve chuva regular entao a produtividade do milho subiu
  3. a fazenda teve chuva regular entao a produtividade do milho depende do solo e
  4. a fazenda teve estiagem severa entao a

---
## Parte 8 — A Linha do Tempo da Memória

Vamos criar uma visualização que mostra, ao longo de uma frase completa, como a "memória" de cada janela vai e vem.

Para cada posição da frase, mostraremos se a palavra-causa ainda está **dentro da janela** do modelo.

In [9]:
def linha_do_tempo_da_memoria(frase_tokens, palavra_causa, janelas):
    """
    Para cada posição na frase, mostra se a palavra-causa ainda está
    visível dentro da janela de cada modelo.

    Legenda:
      ● = palavra-causa está na janela (modelo a 'enxerga')
      ○ = palavra-causa saiu da janela (modelo a 'esqueceu')
      * = esta é a posição da palavra-causa

    Args:
        frase_tokens (list[str]): Tokens da frase.
        palavra_causa (str): A palavra que carrega informação causal.
        janelas (list[int]): Tamanhos de janela a comparar.
    """
    # Encontra a posição da palavra-causa na frase
    try:
        pos_causa = frase_tokens.index(palavra_causa)
    except ValueError:
        print(f"'{palavra_causa}' não encontrada na frase.")
        return

    n_tokens = len(frase_tokens)

    # Cabeçalho: mostra os tokens da frase
    print("Posição: ", end="")
    for i in range(n_tokens):
        marcador = "*" if i == pos_causa else " "
        print(f"{i:>3}{marcador}", end="")
    print()

    print("Token  : ", end="")
    for token in frase_tokens:
        # Trunca tokens longos para caber
        print(f"{token[:4]:>4} ", end="")
    print()

    print()

    # Para cada janela, mostra em quais posições a causa é visível
    for n in janelas:
        print(f"janela={n}: ", end="")
        for pos_atual in range(n_tokens):
            # O modelo está prestes a prever tokens[pos_atual]
            # Ele enxerga tokens[pos_atual-n : pos_atual]
            inicio_visivel = pos_atual - n
            fim_visivel    = pos_atual

            causa_visivel = inicio_visivel <= pos_causa < fim_visivel

            if pos_atual == pos_causa:
                print("  *  ", end="")   # esta posição É a causa
            elif causa_visivel:
                print("  ●  ", end="")   # causa visível
            else:
                print("  ○  ", end="")   # causa invisível
        print()

    print()
    print(f"● = '{palavra_causa}' visível na janela | ○ = fora da janela | * = posição da causa")


print("=" * 70)
print("Linha do tempo: quando 'estiagem' fica invisível para cada modelo")
print("=" * 70)
print()
linha_do_tempo_da_memoria(frase_estiagem, "estiagem", janelas=[1, 2, 3, 5, 7])

Linha do tempo: quando 'estiagem' fica invisível para cada modelo

Posição:   0   1   2   3*  4   5   6   7   8   9  10 
Token  :    a faze teve esti seve enta    a prod   do milh caiu 

janela=1:   ○    ○    ○    *    ●    ○    ○    ○    ○    ○    ○  
janela=2:   ○    ○    ○    *    ●    ●    ○    ○    ○    ○    ○  
janela=3:   ○    ○    ○    *    ●    ●    ●    ○    ○    ○    ○  
janela=5:   ○    ○    ○    *    ●    ●    ●    ●    ●    ○    ○  
janela=7:   ○    ○    ○    *    ●    ●    ●    ●    ●    ●    ●  

● = 'estiagem' visível na janela | ○ = fora da janela | * = posição da causa


---
## Parte 9 — Antes do Transformer: RNN e LSTM

Entre os modelos de n-gramas e o transformer, houve uma geração intermediária: **RNNs e LSTMs**.

Não vamos implementá-los aqui (seriam necessários PyTorch e treinamento), mas vale entender **o que eles tentavam resolver** — e onde ainda falhavam.

---

### RNN — Recurrent Neural Network

A ideia: processar tokens **um a um, em sequência**, mantendo um **estado oculto** `h` que carrega informação do passado.

```
token₁ → [RNN] → h₁
token₂ → [RNN] → h₂  (usa h₁)
token₃ → [RNN] → h₃  (usa h₂)
...                    ↓
tokenₙ → [RNN] → hₙ → predição
```

**Avanço:** ao contrário dos n-gramas, o estado oculto **não tem tamanho fixo de janela** — em teoria, pode acumular toda a história.

**Problema:** na prática, o gradiente que treina a rede **desvanece ou explode** ao se propagar por muitos passos. Informação antiga se perde.

---

### LSTM — Long Short-Term Memory

A solução para o desvanecimento: adicionar **portões** que controlam explicitamente o que lembrar e o que esquecer.

```
portão de esquecimento → decide o que jogar fora da memória
portão de entrada      → decide o que gravar na memória
portão de saída        → decide o que usar da memória agora
```

**Avanço:** melhor memória de longo prazo. Usado em tradução automática, reconhecimento de fala, etc.

**Problema que persiste:**
1. **Natureza sequencial:** processa token a token — impossível paralelizar. Treino lento.
2. **Gargalo:** toda a informação da frase precisa caber em um único vetor `h` de tamanho fixo.
3. **Dependências longas ainda difíceis:** mesmo com LSTM, a informação do token 1 ao alcançar o token 100 foi comprimida muitas vezes.

---

> **Mensagem central:**  
> *"Eles já tentavam guardar contexto, mas ainda pensavam a frase como um fluxo estreito, passo a passo."*

In [10]:
# Simulação conceitual do comportamento de uma RNN
# (não é uma RNN real — é uma simulação do comportamento de memória)
#
# Objetivo: ilustrar o "esvanecimento" da memória ao longo do tempo
# sem precisar de PyTorch ou treinamento.

def simular_memoria_rnn(frase_tokens, fator_esquecimento=0.75):
    """
    Simula o efeito de esvanecimento de memória em uma RNN.

    A cada passo, o 'quanto o token inicial ainda influencia o estado atual'
    é multiplicado pelo fator_esquecimento. Isso modela o problema do
    gradiente desvanecente.

    fator_esquecimento=1.0  → sem esquecimento (memória perfeita)
    fator_esquecimento=0.8  → esquece devagar
    fator_esquecimento=0.5  → esquece rapidamente

    Args:
        frase_tokens (list[str]): Tokens da frase.
        fator_esquecimento (float): Quanto da memória sobrevive a cada passo.

    Returns:
        list[float]: Influência de cada token inicial no estado final.
    """
    n = len(frase_tokens)
    influencias = []

    for i, token in enumerate(frase_tokens):
        # Passos que se passaram desde este token até o final
        passos = n - 1 - i
        # Influência decai exponencialmente
        influencia = fator_esquecimento ** passos
        influencias.append(influencia)

    return influencias


def visualizar_influencia(frase_tokens, fator):
    """Mostra a influência de cada token no estado final da RNN."""
    influencias = simular_memoria_rnn(frase_tokens, fator)

    print(f"Fator de esquecimento = {fator:.2f}")
    print()

    for token, inf in zip(frase_tokens, influencias):
        n_blocos = int(inf * 30)
        barra    = "█" * n_blocos + "░" * (30 - n_blocos)
        print(f"  {token:<18} {barra}  {inf:.3f}")
    print()


print("=" * 65)
print("Influência de cada token no estado final — simulação de RNN")
print("Frase: " + " ".join(frase_estiagem))
print("=" * 65)

print()
print("Com pouco esquecimento (LSTM bem treinada):")
visualizar_influencia(frase_estiagem, fator=0.92)

print("Com esquecimento moderado (RNN típica):")
visualizar_influencia(frase_estiagem, fator=0.75)

print("Com esquecimento forte (RNN em frases longas):")
visualizar_influencia(frase_estiagem, fator=0.55)

print("Observe: 'estiagem' (posição 3) tem influência quase nula no estado final.")
print("É por isso que a RNN ainda falha em dependências longas.")

Influência de cada token no estado final — simulação de RNN
Frase: a fazenda teve estiagem severa entao a produtividade do milho caiu

Com pouco esquecimento (LSTM bem treinada):
Fator de esquecimento = 0.92

  a                  █████████████░░░░░░░░░░░░░░░░░  0.434
  fazenda            ██████████████░░░░░░░░░░░░░░░░  0.472
  teve               ███████████████░░░░░░░░░░░░░░░  0.513
  estiagem           ████████████████░░░░░░░░░░░░░░  0.558
  severa             ██████████████████░░░░░░░░░░░░  0.606
  entao              ███████████████████░░░░░░░░░░░  0.659
  a                  █████████████████████░░░░░░░░░  0.716
  produtividade      ███████████████████████░░░░░░░  0.779
  do                 █████████████████████████░░░░░  0.846
  milho              ███████████████████████████░░░  0.920
  caiu               ██████████████████████████████  1.000

Com esquecimento moderado (RNN típica):
Fator de esquecimento = 0.75

  a                  █░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  0.056
  fazenda   

---
## Parte 10 — Resumo: O Que Aprendemos

Neste notebook demonstramos experimentalmente o **problema do contexto curto**.

---

### O que tentamos

| Abordagem | Contexto | Problema |
|---|---|---|
| Bigrama (n=1) | 1 palavra | Cego para tudo antes da última palavra |
| Trigrama (n=2) | 2 palavras | Pequena melhora local, mesmo problema global |
| n-grama maior | n palavras | Melhora até `n`, depois "esquece" |
| Janela arbitrária | janela fixa | Sempre haverá dependências que excedem a janela |
| RNN | estado oculto acumulado | Gradiente desvanece, memória se comprime |
| LSTM | portões de memória | Melhor, mas ainda sequencial e com gargalo |

---

### O que falta

Todos esses modelos têm uma coisa em comum:

> Eles processam a frase **como um fluxo linear** — do início para o fim, com memória que decai.

O transformer quebra essa suposição:

> **Todo token pode olhar diretamente para qualquer outro token da frase**, sem passar por intermediários.

Isso é o que chamamos de **atenção** — e é o tema do Demo 3.

---

### Visualização final: o contraste

```
N-GRAMA: cada token só vê uma janela fixa atrás
  estiagem  severa  entao  a  produtividade  do  milho  [___]
  ████████  ██████  █████  █  █████████████  ██  █████   ← JANELA=3: só as 3 últimas

RNN/LSTM: cada token carrega um 'resumo' do passado, que vai desbotando
  estiagem  severa  entao  a  produtividade  do  milho  [___]
  ░░░░░░░░  ░░░░░░  ████░  █  █████████████  ██  █████   ← memória decai

TRANSFORMER: cada token olha para TODOS os outros diretamente
  estiagem  severa  entao  a  produtividade  do  milho  [___]
  ████████  ██████  █████  █  █████████████  ██  █████   ← ATENÇÃO GLOBAL
  ↑ 'caiu' ainda consegue ver 'estiagem' diretamente!
```

---

> **Frase para guardar:**  
> *"O transformer mudou o jogo porque trocou uma memória apertada e sequencial por um mecanismo de relação global entre tokens."*

---

## Próximo notebook

No **Demo 3**, vamos implementar e visualizar o mecanismo de **self-attention** — o coração do transformer.  
Veremos como `"caiu"` aprende a olhar diretamente para `"estiagem"`, independente da distância.